In [ ]:
# CARGA DE DATOS
# Importar librerías
import pandas as pd

# Cargar dataset desde raw 
df_original = pd.read_csv("../1_data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Fuente del dataset
fuente = "IBM HR Analytics Employee Attrition & Performance Dataset - Kaggle"
print("Fuente ->", fuente)

# Mostrar primeras filas para confirmar carga
df_original.head()

Fuente -> IBM HR Analytics Employee Attrition & Performance Dataset - Kaggle


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [ ]:
# Limpieza de missing values y duplicados
    # # Columnas tipo int64 y obj que no aportan valor al análisis -> LAS ELIMINO
        # EmployeeCount → siempre tiene valor 1, no aporta información
        # StandardHours → todos tienen 80, no aporta
        # EmployeeNumber → es solo una referencia de identificación, como son datos anómimos no me sirve
        # Over18 → todos son “Y”, no aporta
columnas_eliminar = ['EmployeeCount','StandardHours','EmployeeNumber','Over18']
df_limpia= df_original.drop(columns=columnas_eliminar)

# Hago un shape para ver que ahora mi dataset tiene 31 columnas (he eliminado 4)
df_limpia.shape


(1470, 31)

In [ ]:
# Verifico duplicados exactos

df_limpia.duplicated().sum()  

np.int64(0)

In [ ]:
# Confirmar missing values

df_limpia.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
dtype: int64

In [ ]:
# Para realizar un análisis de correlación, debo convertir Attrition y OverTime a número
# Si no hago esta conversión, me dará error
    # Attrition → Yes=1, No=0
    # OverTime → Yes=1, No=0

df_limpia['Attrition'] = df_limpia['Attrition'].map({'Yes':1, 'No':0}).astype(int)
df_limpia['OverTime'] = df_limpia['OverTime'].map({'Yes':1, 'No':0}).astype(int)

In [ ]:
# Confirmo de nuevo el tipo de datos tras
df_limpia.dtypes

Age                          int64
Attrition                    int64
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
OverTime                     int64
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StockOptionLevel             int64
TotalWorkingYears            int64
TrainingTimesLastYear        int64
WorkLifeBalance              int64
YearsAtCompany               int64
YearsInCurrentRole  

In [ ]:
# Busco algún valor atípico o extremo, en variables que pueden darme outliers

df_limpia[['MonthlyIncome','DistanceFromHome', 'YearsAtCompany','YearsSinceLastPromotion']].describe()

,MonthlyIncome,DistanceFromHome,YearsAtCompany,YearsSinceLastPromotion
count,1470.000000,1470.000000,1470.000000,1470.000000
mean,6502.931293,9.192517,7.008163,2.187755
std,4707.956783,8.106864,6.126525,3.222430
min,1009.000000,1.000000,0.000000,0.000000
25%,2911.000000,2.000000,3.000000,0.000000
50%,4919.000000,7.000000,5.000000,1.000000
75%,8379.000000,14.000000,9.000000,3.000000
max,19999.000000,29.000000,40.000000,15.000000


In [ ]:
# Selecciono solo variables numéricas con mas de 2 valores unicos
num_vars = df_limpia.select_dtypes(include=['int64']).columns
num_vars = [var for var in num_vars if df_limpia[var].nunique() > 2]

outliers_num_count = {}
for var in num_vars:
    Q1 = df_limpia[var].quantile(0.25)
    Q3 = df_limpia[var].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = df_limpia[(df_limpia[var] < lower) | (df_limpia[var] > upper)][var].shape[0]
    if count > 0:
        outliers_num_count[var] = count

print("Variables con outliers (excluyendo binarias):")
print(outliers_num_count)

"""
DECISIÓN SOBRE OUTLIERS:
He identificado outliers en varias variables (MonthlyIncome, YearsAtCompany, etc.).
Sin embargo, he decididoMANTENERLOS porque:
1. Pueden representar casos reales (ejecutivos con salarios altos, empleados muy antiguos)
2. Quiero preservar toda la información para el análisis exploratorio
3. En un análisis posterior podría considerar tratarlos
"""

Variables con outliers (excluyendo binarias):
{'MonthlyIncome': 114, 'NumCompaniesWorked': 52, 'StockOptionLevel': 85, 'TotalWorkingYears': 63, 'TrainingTimesLastYear': 238, 'YearsAtCompany': 104, 'YearsInCurrentRole': 21, 'YearsSinceLastPromotion': 107, 'YearsWithCurrManager': 14}


'\nDECISIÓN SOBRE OUTLIERS:\nHe identificado outliers en varias variables (MonthlyIncome, YearsAtCompany, etc.).\nSin embargo, decidí MANTENERLOS porque:\n1. Pueden representar casos reales (ejecutivos con salarios altos, empleados muy antiguos)\n2. Quiero preservar toda la información para el análisis exploratorio\n3. En un análisis posterior (modelado) podría considerar tratarlos\n'

In [ ]:
# Ahora hago un análisis de rangos lógicos, para detectar si hay algún error extremo

def chequeo_rapido_errores_df_limpia(df_limpia):
    """Versión adaptada para tu df_limpia - 2 minutos"""
    variables_criticas = ['PerformanceRating', 'StockOptionLevel', 
                         'TrainingTimesLastYear', 'YearsAtCompany',
                         'YearsSinceLastPromotion', 'NumCompaniesWorked']
    
    print("=== ANÁLISIS DE VALORES EN DF_LIMPIA ===")
    for var in variables_criticas:
        if var in df_limpia.columns:  # Verifico que existe en el dataset
            print(f"\n{var}:")
            print(f"   Únicos: {sorted(df_limpia[var].unique())}")
            print(f"   Min: {df_limpia[var].min()}, Max: {df_limpia[var].max()}")
            
            # Verifico escalas
            if var in ['PerformanceRating', 'StockOptionLevel']:
                if df_limpia[var].max() > 4:
                    print(f"   ERROR: {var} debería ser 1-4, pero tiene {df_limpia[var].max()}")
            elif var == 'TrainingTimesLastYear':
                if df_limpia[var].max() > 20:
                    print(f"   ERROR: {var} máximo razonable 20, pero tiene {df_limpia[var].max()}")
        else:
            print(f"\n{var}: NO EXISTE en df_limpia")

chequeo_rapido_errores_df_limpia(df_limpia)

=== ANÁLISIS DE VALORES EN DF_LIMPIA ===

PerformanceRating:
   Únicos: [np.int64(3), np.int64(4)]
   Min: 3, Max: 4

StockOptionLevel:
   Únicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
   Min: 0, Max: 3

TrainingTimesLastYear:
   Únicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
   Min: 0, Max: 6

YearsAtCompany:
   Únicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(37), np.int64(40)]
   Min: 0, Max: 40

YearsSinceLastPromotion:
   Únicos: [np.int64(0), np.int64(1), np.int64(2), np.i

In [ ]:
# Hago un análisis de consistencia de valores numéricos

def consistencia_rapida_df_limpia(df_limpia):
    """Versión adaptada para tu dataset limpio - 1 minuto"""
    problemas = []
    
    # Verifico que las variables necesarias existen
    variables_necesarias = ['YearsSinceLastPromotion', 'YearsAtCompany', 'TotalWorkingYears', 'Age']
    variables_faltantes = [var for var in variables_necesarias if var not in df_limpia.columns]
    
    if variables_faltantes:
        print(f"Variables faltantes para consistencia: {variables_faltantes}")
        return problemas
    
    # Años sin promoción no puede ser > años en empresa
    if (df_limpia['YearsSinceLastPromotion'] > df_limpia['YearsAtCompany']).any():
        count = (df_limpia['YearsSinceLastPromotion'] > df_limpia['YearsAtCompany']).sum()
        problemas.append(f"YearsSinceLastPromotion > YearsAtCompany: {count} casos")
    
    # Años en empresa no puede ser > años trabajando total
    if (df_limpia['YearsAtCompany'] > df_limpia['TotalWorkingYears']).any():
        count = (df_limpia['YearsAtCompany'] > df_limpia['TotalWorkingYears']).sum()
        problemas.append(f"YearsAtCompany > TotalWorkingYears: {count} casos")
    
    # Edad razonable para trabajar
    if (df_limpia['Age'] < 16).any() or (df_limpia['Age'] > 80).any():
        count_menor = (df_limpia['Age'] < 16).sum()
        count_mayor = (df_limpia['Age'] > 80).sum()
        problemas.append(f"Edad extrema (<16: {count_menor}, >80: {count_mayor})")
    
    return problemas

print("\n=== 🔧 VERIFICACIÓN DE CONSISTENCIA EN DF_LIMPIA ===")
problemas = consistencia_rapida_df_limpia(df_limpia)
if problemas:
    print("Problemas encontrados:")
    for p in problemas:
        print(f"   - {p}")
else:
    print("No se encontraron problemas de consistencia")


=== 🔧 VERIFICACIÓN DE CONSISTENCIA EN DF_LIMPIA ===
No se encontraron problemas de consistencia


In [ ]:
"""
FASE 2.4 - VERIFICACIÓN DE CALIDAD Y CONSISTENCIA DEL DATASET

OBJETIVO:
Asegurarme de que toda la información esté correcta y tenga sentido antes de analizarla.

QUÉ REVISÉ:
1. Los números en cada columna:
   - Que las evaluaciones de desempeño vayan de 1 a 4 (y sí, van de 3 a 4)
   - Que los años en la empresa sean números realistas (sí, de 0 a 40 años)
   - Que los entrenamientos por año sean posibles (sí, de 0 a 6)

2. Las relaciones entre datos:
   - Que los años sin promoción no sean más que los años en la empresa 
   - Que la edad de los empleados sea razonable para trabajar 
   - Que toda la información esté lógicamente conectada

RESULTADO:
No he encontrado errores ni datos que no tuvieran sentido.

DECISIÓN:
Voy a trabajar con una copia de estos datos ya validados para mantener todo organizado.
"""

df_corregida = df_limpia.copy()
 

In [ ]:
# Crear copia final para análisis
df_corregida = df_limpia.copy()

# Guardar dataset limpio y validado
df_corregida.to_csv("../1_data/processed/df_corregida.csv", index=False)
print("Dataset limpio guardado como df_corregida.csv")

✅ Dataset limpio guardado como df_corregida.csv
